In [37]:
import os
from dotenv import load_dotenv, find_dotenv

from utils import RAGRetriever, EmbeddingManager, VectorStore

from langchain_groq import ChatGroq

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

### **Augmentation**

In [ ]:
def simple_rag(query, retriever: RAGRetriever, llm, top_k=3):
    """
    Retrieves context from vector store and augments to LLM with the original query

    Args:
        query: original query from the user
        retriever: RAGRetriever class which retrieves docs from the vector store
        llm: model that generates output
        top_k: Number of top results to return
    """

    results = retriever.retrieve(query=query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question:
        {query}

        Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])

    return response.content    

In [35]:
retriever = RAGRetriever(VectorStore(), EmbeddingManager())
answer = simple_rag(query="What is automation and how it works", retriever=retriever, llm=llm)

print(answer)

Vector store initialized. Collection: pdf_documents
Exisiting documents in collection: 114
Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2059.24it/s]


Model loaded successfully. Embedding dimension: 384
Retrieving documents for the query What is automation and how it works
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.38it/s]

Generated embeddings with shape (1, 384)
Retrieved 1 documents
[{'id': 'doc_e41d38e2_25', 'content': '# **-** **Practical  5** **Aim: Understand the Automation Testing approach** **(Theory concept)**\n\nAutomation:\n\n\nAutomation is making a process automatic eliminating the need for\nhuman intervention. It is a self-controlling or self-moving process.\n\nAutomation Software offers automation wizards and commands of\n\nits own in addition to providing a task recording and re-play\ncapabilities. Using these programs, you can record an IT or business\n\ntask.\n\n\nBenefits of Automation\n\n\n    - Fast\n\n\n    - Reliable\n\n\n    Repeatable\n\n\n    Programmable\n\n\n    - Reusable\n\n\n    Makes Regression testing easy\n\n\n    Enables 24*78 Testing\n\n\n    - Robust verification.\n\n\n# **i**', 'metadata': {'subject': '', 'creationDate': '', 'format': 'PDF 1.7', 'modDate': 'D:20260311171953Z', 'file_path': '..\\..\\data\\pdf\\STdisha5-6.pdf', 'page': 0, 'source': '..\\..\\data\\pdf\\

<think>
Okay, the user is asking about automation and how it works. Let me start by recalling the context provided. The context mentions that automation is making a process automatic without human intervention, using software with wizards, commands, recording, and replay capabilities.

First, I need to define automation clearly. The context says it's a self-controlling or self-moving process. So, I should explain that automation eliminates the need for manual input. Then, how it works: the context talks about automation software using recording and replay features. Maybe mention that tasks are recorded once and then replayed automatically. Also, the software has its own commands and wizards to create automated processes.

I should structure the answer into two parts: definition and mechanism. Make sure to keep it concise as per the user's request. Check the benefits listed in the context, but the question is about how it works, so focus on the process. Avoid listing benefits unless nec

### **Enhanced RAG Pipeline features**

In [38]:
def advanced_rag(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    Returns answers, sources, confidence score and optionally full context
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {"answer": "No relevant context found.", "sources": [], "confidence": 0.0, "context": "" }
    
    context = "\n\n".join([doc["content"] for doc in results])

    sources = [{
        "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
        "page": doc["metadata"].get("page", "unknown"),
        "score": doc["similarity_score"],
        "preview": doc["content"][:300] + "..."
    } for doc in results]
    confidence = max([doc["similarity_score"] for doc in results])

    prompt = f"""Use the following context to answer  the question concisely. \nContext:\n{context}\n\nQuestion:\n{query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        output["context"] = context
    
    return output

In [40]:
result = advanced_rag(
    query="What is automation and how it works",
    retriever=retriever,
    llm=llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer: ", result["answer"])
print("Sources: ", result["sources"])
print("Confidence: ", result["confidence"])
print("Context: ", result["context"][:300])

Retrieving documents for the query What is automation and how it works
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.65it/s]

Generated embeddings with shape (1, 384)
Retrieved 1 documents


Answer:  Automation is a self-controlling process that eliminates human intervention, making a process automatic. It works by using automation software that offers wizards, commands, task recording, and replay capabilities, allowing users to record and repeat tasks.
Sources:  [{'source': '..\\..\\data\\pdf\\STdisha5-6.pdf', 'page': 0, 'score': 0.4506782293319702, 'preview': '# **-** **Practical  5** **Aim: Understand the Automation Testing approach** **(Theory concept)**\n\nAutomation:\n\n\nAutomation is making a process automatic eliminating the need for\nhuman intervention. It is a self-controlling or self-moving process.\n\nAutomation Software offers automation wizards and c...'}]
Confidence:  0.4506782293319702
Context:  # **-** **Practical  5** **Aim: Understand the Automation Testing approach** **(Theory concept)**

Automation:


Automation is making a process automatic eliminating the need for
human intervention. It is a self-controlling or self-moving process.

Automation Softw